# 模型評估方法實驗:交叉驗證、混淆矩陣、Precision/Recall 與 ROC

**這份筆記在做什麼?**

前面幾份實驗都在「訓練模型」;這一份回答更根本的問題 —— **模型到底好不好,怎麼量?**
你會看到:光看「準確率」有多容易被騙,以及一整套更可靠的評估工具。

全篇用同一個任務貫穿:MNIST 手寫數字的**二分類 ——「這張圖是不是 5」**。

**學習地圖:**

| 部分 | 主題 | 回答的問題 |
|:---:|---|---|
| 1 | 資料準備 | MNIST 載入、切分、洗牌 |
| 2 | 交叉驗證 | 只切一次訓練/測試,夠可靠嗎? |
| 3 | 混淆矩陣 | 準確率為什麼會騙人?錯誤分哪幾種? |
| 4 | Precision / Recall / F1 | 「抓得準」和「抓得全」怎麼量? |
| 5 | 閾值的影響 | 移動決策門檻,會發生什麼事? |
| 6 | ROC 曲線與 AUC | 一條曲線、一個數字總結模型好壞 |

> 💡 請**由上往下依序執行**每個 cell(Shift + Enter)。交叉驗證的 cell 需要跑數十秒,屬正常現象。

## 0. 環境準備

In [1]:
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from sklearn import set_config

# Windows 中文環境可能以 CP950 讀取 scikit-learn 的 UTF-8 HTML 資源而失敗
# 改用文字表示不影響模型訓練、預測或評估結果
set_config(display='text')

# 統一圖表字體大小
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

import warnings
warnings.filterwarnings('ignore')   # 忽略警告訊息,保持輸出乾淨
np.random.seed(42)                  # 固定隨機種子,結果可重現

## 1. 資料集讀取

MNIST 是影像資料:70,000 張 28×28 的**灰階手寫數字圖**,每張攤平成 784 個像素值(0~255),
標籤是 0~9。長這樣:

![MNIST 手寫數字樣本](./img/9.png)

In [2]:
from sklearn.datasets import fetch_openml

# fetch_mldata 已在新版 scikit-learn 移除,改用 OpenML 載入 MNIST(第一次會下載,之後讀快取)
mnist_openml = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
mnist = {
    'data': mnist_openml['data'],
    'target': mnist_openml['target'].astype(float),
    'DESCR': 'OpenML dataset: mnist_784',
    'COL_NAMES': ['label', 'data'],
}

In [3]:
X, y = mnist["data"], mnist["target"]
X.shape   # 70000 張圖,每張 784 個像素

(70000, 784)

In [4]:
y.shape   # 70000 個標籤(0~9)

(70000,)

### 1.2 切分與洗牌

MNIST 慣例:**前 60,000 張當訓練集、後 10,000 張當測試集**。
另外把訓練集**洗牌(shuffle)**:如果資料按某種順序排列(例如同數字連在一起),
交叉驗證切出來的每一折分布會不均勻,影響評估的公正性。

In [5]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

In [6]:
# 洗牌操作:產生 0~59999 的隨機排列,把訓練資料重新洗過
shuffle_index = np.random.permutation(60000)
X_train, y_train = X_train[shuffle_index], y_train[shuffle_index]

In [7]:
shuffle_index   # 洗牌後的索引順序

array([12628, 37730, 39991, ...,   860, 15795, 56422],
      shape=(60000,), dtype=int32)

## 2. 交叉驗證(Cross Validation)

評估模型不能用「模型看過的資料」。基本做法是切訓練/測試集,資料夠的話再多切一塊驗證集:

![資料集切分:訓練/測試/驗證與交叉驗證](./img/5.png)

但只切一次,分數會受「切在哪裡」的運氣影響。**交叉驗證**把訓練集切成 k 折(下圖 k=5):
輪流拿其中 1 折當驗證、其餘 k−1 折訓練,共得到 k 個分數 —— 更穩、更公正:

![5 折交叉驗證:每一折輪流當驗證集](./img/7.png)

### 2.1 建立實驗任務:「是不是 5」

把 10 類問題簡化成二分類:標籤是 5 → True,其他 → False。

In [8]:
y_train_5 = (y_train == 5)   # 標籤變換:是 5 → True,不是 → False
y_test_5 = (y_test == 5)

In [9]:
y_train_5[:10]   # 大多是 False(畢竟 5 只佔十分之一)

array([False, False, False, False, False, False, False, False,  True,
       False])

### 2.2 訓練一個分類器

用 `SGDClassifier`:一個用**隨機梯度下降**(線性迴歸篇的老朋友)訓練的線性分類器,
訓練快,適合大資料。`max_iter=5` 表示最多掃 5 輪資料:

In [10]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(max_iter=5, random_state=42)
sgd_clf.fit(X_train, y_train_5);  # 分號避免 IDE 嘗試產生有編碼問題的 HTML 模型圖

拿一張**真的是 5** 的圖來試(`X[0]` 恰好就是一張 5,先畫出來確認):

In [ ]:
plt.imshow(X[0].reshape(28, 28), cmap=matplotlib.cm.binary)   # 把 784 維向量排回 28×28
plt.axis('off')
plt.show(block=False)
print('這張圖的真實標籤:', y[0])

In [ ]:
sgd_clf.predict([X[0]])   # 模型認得出這張 5 嗎?

這張抓到了。多拿幾張真的 5 來測測看:

In [ ]:
five_indices = np.where(y == 5.0)[0][:10]   # 找出前 10 張真的是 5 的圖
sgd_clf.predict(X[five_indices])            # 10 張裡抓到幾張?

10 張真 5 只抓到 6 張 —— 先記住這個現象,第 4 部分的「召回率」就是在量它。

### 2.3 用 cross_val_score 做交叉驗證

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring='accuracy')   # 3 折交叉驗證的準確率

In [ ]:
X_train.shape

In [ ]:
y_train_5.shape

三折準確率都在 **96% 左右**,看起來很棒?

> ⚠️ **準確率陷阱**:「5」只佔全部資料的約 10% ——
> 一個**什麼都不學、永遠回答「不是 5」**的模型,準確率也有 90%!
> 在類別不平衡的任務上,高準確率可能什麼都沒說明 —— 這正是本篇要引入其他評估工具的原因。

### 2.4 交叉驗證的內部原理:手動實作一次

`cross_val_score` 幫我們做掉的事,自己寫一遍就懂了。`StratifiedKFold` 做**分層抽樣**:
每一折裡「是 5 / 不是 5」的比例都和整體一致:

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

skflods = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for train_index, test_index in skflods.split(X_train, y_train_5):
    clone_clf = clone(sgd_clf)                    # 複製一個「還沒訓練」的同款模型
    X_train_folds = X_train[train_index]          # 這一輪的訓練折
    y_train_folds = y_train_5[train_index]
    X_test_folds = X_train[test_index]            # 這一輪的驗證折
    y_test_folds = y_train_5[test_index]

    clone_clf.fit(X_train_folds, y_train_folds)   # 訓練 → 驗證 → 算準確率
    y_pred = clone_clf.predict(X_test_folds)
    n_correct = sum(y_pred == y_test_folds)
    print(n_correct / len(y_pred))

## 3. Confusion Matrix - 混淆矩陣

要看穿準確率的假象,得把「錯誤」拆開來看。二分類的每筆預測只有四種結局
(下圖用「從班上找出所有女生」的例子說明,TP=真陽性、FP=偽陽性、FN=偽陰性、TN=真陰性):

![TP / FP / FN / TN 四種結局](./img/8.png)

### 3.1 先拿到「乾淨」的預測:cross_val_predict

要算混淆矩陣,每筆資料都需要一個預測值 —— 但不能用「模型看過它」時的預測。
`cross_val_predict` 同樣切 3 折:每筆資料的預測,都來自**沒看過它**的那個模型:

In [ ]:
from sklearn.model_selection import cross_val_predict
y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

In [ ]:
y_train_pred.shape   # 60000 筆,每筆都有一個「乾淨」的預測

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_train_5, y_train_pred)

**怎麼讀這個 2×2 矩陣**(列 = 真實類別,欄 = 預測類別):

```
                 預測「不是5」   預測「是5」
真實不是5  [[ true negatives , false positives ],
真實是5    [  false negatives, true positives  ]]
```

以這次的結果為例:

* **true negatives(54,058)**:不是 5、也被正確判為不是 5
* **false positives(521)**:不是 5,卻被誤判成 5
* **false negatives(1,899)**:明明是 5,卻被漏判成不是 5
* **true positives(3,522)**:是 5、也被正確抓出來

一個**完美的分類器**只有 true positives 和 true negatives —— 主對角線不為 0,其餘位置全是 0。

## 4. Precision 與 Recall

從混淆矩陣提煉出兩個最重要的指標:

### $precision = \dfrac{TP}{TP + FP}$ —— 模型說「是 5」的那些,有多少**真的是 5**?(抓得準不準)

### $recall = \dfrac{TP}{TP + FN}$ —— 所有真的 5 裡,模型**抓到了幾成**?(抓得全不全)

![混淆矩陣與 Precision / Recall 的方向](./img/1.png)

In [ ]:
from sklearn.metrics import precision_score, recall_score
precision_score(y_train_5, y_train_pred)   # 3522 / (3522 + 521)

In [ ]:
recall_score(y_train_5, y_train_pred)   # 3522 / (3522 + 1899)

precision ≈ 0.87:模型喊「是 5」時,87% 是對的;
recall ≈ 0.65:所有真 5 裡只抓到 65% —— 和 2.2 節「10 張抓到 6 張」完全呼應。

將 **Precision** 和 **Recall** 結合成一個指標 —— **F1 score**:兩者的**調和平均**。
調和平均給低值更多權重,所以只有當 precision 和 recall **都高**時,F1 才會高:

### $F_1 = \dfrac{2}{\frac{1}{precision} + \frac{1}{recall}} = 2 \times \dfrac{precision \times recall}{precision + recall} = \dfrac{TP}{TP + \frac{FN + FP}{2}}$

In [ ]:
from sklearn.metrics import f1_score
f1_score(y_train_5, y_train_pred)

## 5. 閾值對結果的影響

Precision 和 Recall 為什麼一高一低?因為模型內部其實是給每張圖打一個**分數**,
再用一個**閾值(threshold)**一刀切:分數 > 閾值 → 判為 5。

**閾值往右移**:出手變保守 → precision 升、recall 降;**往左移**則相反 —— 魚與熊掌:

![移動閾值:precision 與 recall 的取捨](./img/2.png)

### 5.1 單張示範:decision_function

Scikit-Learn 不允許直接設定閾值,但可以呼叫 **`decision_function()`** 取得每個樣本的決策分數,
再用自己想要的閾值做判斷(`predict()` 其實就等於「分數 > 0」):

In [ ]:
y_scores = sgd_clf.decision_function([X[0]])   # 那張 5 的決策分數
y_scores

In [ ]:
t = 0                    # 閾值 = 0:等同於 predict() 的預設行為
y_pred = (y_scores > t)
y_pred

In [ ]:
t = 50000                # 把閾值調高到超過這張圖的分數
y_pred = (y_scores > t)
y_pred                    # 同一張 5,就這樣被「漏掉」了 → recall 下降

### 5.2 對全部樣本取分數,畫出取捨曲線

用 `cross_val_predict(method="decision_function")` 拿到 60,000 筆的「乾淨」分數:

In [ ]:
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3,
                             method="decision_function")

In [ ]:
y_scores[:10]   # 分數越大,模型越覺得「是 5」

In [ ]:
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

In [ ]:
y_train_5.shape

In [ ]:
thresholds.shape   # 59999 個候選閾值

In [ ]:
precisions[:10]

In [ ]:
precisions.shape   # precision / recall 各 60000 個(比閾值多 1,對應「全判為正」的端點)

In [ ]:
recalls.shape

把「每個閾值下的 precision 和 recall」畫在同一張圖:

In [ ]:
def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):
    plt.plot(thresholds,
             precisions[:-1],
            "b--",
            label="Precision")

    plt.plot(thresholds,
             recalls[:-1],
            "g-",
            label="Recall")
    plt.xlabel("Threshold", fontsize=16)
    plt.legend(loc="upper left", fontsize=16)
    plt.ylim([0, 1])

plt.figure(figsize=(8, 4))
plot_precision_recall_vs_threshold(precisions, recalls, thresholds)
plt.xlim([-700000, 700000])
plt.show(block=False)

**觀察結果:** 閾值往右調,precision(藍)一路上升、recall(綠)一路下降 —— 完全的此消彼長。
實務上就是在這張圖上,依任務需求挑一個取捨點
(寧可錯殺不可放過 → 壓低閾值救 recall;寧缺勿濫 → 拉高閾值救 precision)。

也可以直接畫 **Precision 對 Recall**:

In [ ]:
def plot_precision_vs_recall(precisions, recalls):
    plt.plot(recalls,
             precisions,
             "b-",
             linewidth=2)

    plt.xlabel("Recall", fontsize=16)
    plt.ylabel("Precision", fontsize=16)
    plt.axis([0, 1, 0, 1])

plt.figure(figsize=(8, 6))
plot_precision_vs_recall(precisions, recalls)
plt.show(block=False)

**觀察結果:** recall 超過 0.6 之後 precision 開始跳水 —— 一般會把工作點選在「懸崖」之前。

## 6. ROC 曲線

**Receiver Operating Characteristic(ROC)曲線**是二元分類中的常用評估方法:

* 它與 precision/recall 曲線非常相似,但 ROC 曲線畫的是 **true positive rate(TPR)** 對
  **false positive rate(FPR)**
* 要繪製 ROC 曲線,先用 **`roc_curve()`** 計算各種閾值下的 TPR 和 FPR:

$TPR = \dfrac{TP}{TP + FN}$(就是 Recall)

$FPR = \dfrac{FP}{FP + TN}$(不是 5 的圖裡,被誤判成 5 的比例)

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)

In [ ]:
def plot_roc_curve(fpr, tpr, label=None):
    plt.plot(fpr, tpr, linewidth=2, label=label)
    plt.plot([0, 1], [0, 1], 'k--')   # 對角虛線:純隨機分類器
    plt.axis([0, 1, 0, 1])
    plt.xlabel('False Positive Rate', fontsize=16)
    plt.ylabel('True Positive Rate', fontsize=16)

plt.figure(figsize=(8, 6))
plot_roc_curve(fpr, tpr)
plt.show(block=False)

**虛線表示純隨機分類器的 ROC 曲線**;一個好的分類器要盡可能遠離該線(往**左上角**貼)。

比較分類器的常用方法是量測**曲線下面積(AUC)**:完美分類器的 ROC AUC **等於 1**,
純隨機分類器的 ROC AUC **等於 0.5**。Scikit-Learn 直接提供:

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_train_5, y_scores)

AUC ≈ 0.96,離隨機的 0.5 很遠 —— 模型的「排序能力」不錯。

> 📝 挑選曲線的小訣竅:**正類很稀有、或你更在乎 FP/FN 的代價時,看 PR 曲線**(像本例的「找 5」);
> 類別大致平衡時,ROC/AUC 是通用的預設選擇。

## 總結:這份實驗的重點整理

| # | 重點 |
|:---:|---|
| 1 | 評估一定要用模型**沒看過**的資料;**交叉驗證**(k 折輪流當驗證)比切一次更可靠 |
| 2 | **準確率會騙人**:類別不平衡時,全猜多數類就有高準確率(「找 5」全猜否也有 90%) |
| 3 | **混淆矩陣**把錯誤拆成 TP/FP/FN/TN;`cross_val_predict` 提供「乾淨」的預測 |
| 4 | **Precision = 抓得準;Recall = 抓得全;F1 = 兩者的調和平均**(都高才高) |
| 5 | 模型輸出的是分數,**閾值決定取捨**:閾值高 → precision 升、recall 降 |
| 6 | **ROC 曲線**畫 TPR vs FPR,**AUC** 一個數字總結(1=完美,0.5=隨機);正類稀有時優先看 PR 曲線 |

**多做實驗,得出結果!!!**